<font color='blue' size=5><b>Комментарий ревьюера</b></font>

<font color='blue'>Привет, Оксана! Меня зовут Павел Григорьев, и я буду проверять этот проект.<br>Моя основная цель - не указать на совершённые тобой ошибки, а поделиться своим опытом и помочь тебе совершенствоваться как профессионалу.<br>Спасибо за проделанную работу! Предлагаю общаться на «ты».</font>
<details>
	<summary><u>Инструкция по организационным моментам (кликабельно)</u></summary>
<font color='blue'>Я буду использовать различные цвета, чтобы было удобнее воспринимать мои комментарии:</font>


---


<font color='blue'>синий текст - просто текст комментария</font>

<font color='green'>✔️ и зеленый текст - все отлично</font>

<font color='orange'>⚠️ и оранжевый текст - сделано все правильно, однако есть рекомендации, на что стоит обратить внимание</font>

<font color='red'>❌ и красный текст - есть недочеты</font>


</details>    
    </br>
<font color='blue'>Пожалуйста, не удаляй мои комментарии в случае возврата работы, так будет проще разобраться, какие были недочеты, а также сразу увидеть исправленное. </font>

Ответы на мои комментарии лучше тоже помечать.
Например: <font color='purple'><b>Комментарий студента</b></font>

<font color='blue'><b>Давай смотреть, что получилось!</b></font>

<h1>Содержание<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Загрузка-данных" data-toc-modified-id="Загрузка-данных-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Загрузка данных</a></span></li><li><span><a href="#Подготовка" data-toc-modified-id="Подготовка-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Подготовка</a></span></li><li><span><a href="#Обучение" data-toc-modified-id="Обучение-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Обучение</a></span></li><li><span><a href="#Выводы" data-toc-modified-id="Выводы-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Выводы</a></span></li><li><span><a href="#Чек-лист-проверки" data-toc-modified-id="Чек-лист-проверки-5"><span class="toc-item-num">5&nbsp;&nbsp;</span>Чек-лист проверки</a></span></li></ul></div>

# Проект для «Викишоп»

Интернет-магазин «Викишоп» запускает новый сервис. Теперь пользователи могут редактировать и дополнять описания товаров, как в вики-сообществах. То есть клиенты предлагают свои правки и комментируют изменения других. Магазину нужен инструмент, который будет искать токсичные комментарии и отправлять их на модерацию. 

Обучите модель классифицировать комментарии на позитивные и негативные. В вашем распоряжении набор данных с разметкой о токсичности правок.

Постройте модель со значением метрики качества *F1* не меньше 0.75. 

**Инструкция по выполнению проекта**

1. Загрузите и подготовьте данные.
2. Обучите разные модели. 
3. Сделайте выводы.

Для выполнения проекта применять *BERT* необязательно, но вы можете попробовать.

**Описание данных**

Данные находятся в файле `toxic_comments.csv`. Столбец *text* в нём содержит текст комментария, а *toxic* — целевой признак.

## Загрузка данных

In [24]:
!pip install --upgrade pip
!pip install --upgrade scikit-learn -q
!pip install nltk -q
!pip install wordcloud -q
!pip install pandas -q
!pip install lightgbm -q
!pip install -U textblob -q

In [25]:
import pandas as pd
import numpy as np

from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
from nltk.corpus import wordnet
from nltk.corpus import stopwords as nltk_stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split, KFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
import lightgbm as lgb
from textblob import TextBlob

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import f1_score

from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud, STOPWORDS

import matplotlib.pyplot as plt
%matplotlib inline

import re
from tqdm import notebook
notebook.tqdm.pandas()

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/226/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /Users/226/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/226/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /Users/226/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


<font color='blue'><b>Комментарий ревьюера: </b></font> ✔️\
<font color='green'> Хорошее оформление импортов! \
Импорты собраны в одной ячейке, разделены на функциональные группы пустой строкой.</font>

In [26]:
try:
    data = pd.read_csv('/datasets/toxic_comments.csv')
except:
    data = pd.read_csv('https://code.s3.yandex.net/datasets/toxic_comments.csv')

In [27]:
def analyze_dataframe(data):

    missed_cells = data.isnull().sum().sum()/(data.shape[0]*(data.shape[1]-1))
    missed_rows = sum(data.isnull().sum(axis = 1)>0)/data.shape[0]
    print ('\033[1m' + '\nПроверка пропусков'+ '\033[0m')
    print ('Количество пропусков: {:.0f}'.format(data.isnull().sum().sum()))
    print ('Доля пропусков: {:.1%}'.format(missed_cells)+ '\033[0m')
    print ('Доля строк содержащих пропуски: {:.1%}'.format(missed_rows))

    ## Проверим дубликаты
    print ('\033[1m' + '\nПроверка на дубликаты'+ '\033[0m')
    print('Количество явных дубликатов: ', data.duplicated().sum())


    # Вывод основных статистических характеристик датафрейма

    print('\033[1m' + '\n0писание количественных данных:'+ '\033[0m')
    display(data.describe())


    # Вывод общей информации о датафрейме
    print('\033[1m' + '\nInfo:'+ '\033[0m')
    display(data.info())

    # Приведение колонок к нижнему регистру
    data.columns = data.columns.str.lower()

<font color='blue'><b>Комментарий ревьюера: </b></font> ⚠️\
<font color='darkorange'> Метод info() не нуждается в display(), и возвращает None.</font>

<div class="alert alert-block alert-warning">
<b>Комментарий студента:</b> <br>
<b>Изменения:</b> Были внесены следующие изменения: display() удален
</div>

In [28]:
display(analyze_dataframe(data))


Проверка пропусков
Количество пропусков: 0
Доля пропусков: 0.0%
Доля строк содержащих пропуски: 0.0%

Проверка на дубликаты
Количество явных дубликатов:  0

0писание количественных данных:


,Unnamed: 0,toxic
count,159292.000000,159292.000000
mean,79725.697242,0.101612
std,46028.837471,0.302139
min,0.000000,0.000000
25%,39872.750000,0.000000
50%,79721.500000,0.000000
75%,119573.250000,0.000000
max,159450.000000,1.000000



Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159292 entries, 0 to 159291
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   Unnamed: 0  159292 non-null  int64 
 1   text        159292 non-null  object
 2   toxic       159292 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.6+ MB


None

None

In [29]:
display(data['toxic'].value_counts())
display(data['toxic'].value_counts(normalize=True).map('{:.2%}'.format))

toxic
0    143106
1     16186
Name: count, dtype: int64

toxic
0    89.84%
1    10.16%
Name: proportion, dtype: object

<font color='blue'><b>Комментарий ревьюера: </b></font> ✔️\
<font color='green'> Мы обнаружили серьёзный дисбаланс при исследовании данных. Как дополнительные материалы, рекомендую статью <a href='https://dyakonov.org/2021/05/27/imbalance/'>Дисбаланс классов</a>, очень классная, как и весь блог Дьяконова. Ещё такой <a href='https://github.com/Dyakonov/ml_hacks/blob/master/book_disbalance_public_v1.ipynb'>ноутбук</a> есть.</font>

In [30]:
# посмотрим на случайные объекты
display(data.sample(5, random_state=22))
for comment in data.text.sample(5, random_state=22):
    print(100 * '_' + '\n' + comment)

,unnamed: 0,text,toxic
111890,111987,(1) Not that I can think of. (2) I'm not sure....,0
80864,80940,"""\n\n Email \n\nHiya,\n\nEmail for you. Can yo...",0
103344,103441,If what you've found isn't WP:OR then please g...,0
122353,122458,"""\n\n Please do not vandalize pages, as you di...",0
58810,58876,Danny Green article peer review \n\nThis artic...,0


____________________________________________________________________________________________________
(1) Not that I can think of. (2) I'm not sure. What good do you think this will do? You can't cite personal communications here.
____________________________________________________________________________________________________
"

 Email 

Hiya,

Email for you. Can you let me know when you get it?

Many thanks! FT2 (Talk | email) "
____________________________________________________________________________________________________
If what you've found isn't WP:OR then please go ahead and add the reference. If you have some pieces that need an OR synthesis in a research paper then wikipedia isn't the place. hth
____________________________________________________________________________________________________
"

 Please do not vandalize pages, as you did with this edit to Alberto Tomás Botía Rabasco. If you continue to do so, you will be blocked from editing.   /talk to me/ "
________

 <span style="color:#8b51f6">**Вывод**</span> &#x2705; <br>
 
&#x1F31F; явных дубликатов не обнаружено <br>
&#x1F31F; пропусков не обнаружено <br>
&#x1F31F; присутствует дисбаланс классов в целевой переменной <br>
&#x1F31F; требуется очистка данных (спецсимволы, знаки препинания) <br>
&#x1F31F; требуется привести текст к нижнему регистру<br>

<font color='blue'><b>Комментарий ревьюера: </b></font> ✔️\
<font color='green'>Данные загружены корректно, первичный осмотр проведен.</font>

## Подготовка

In [31]:
# уберем пунктуацию, произведем замены и приведем текст в нижний регистр
def remove_odd(text):
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    text = re.sub(r"what's", "what is ", text)
    text = re.sub(r"\'s", " ", text)
    text = re.sub(r"\'ve", " have ", text)
    text = re.sub(r"can't", "cannot ", text)
    text = re.sub(r"n't", " not ", text)
    text = re.sub(r"i'm", "i am ", text)
    text = re.sub(r"\'re", " are ", text)
    text = re.sub(r"\'d", " would ", text)
    text = re.sub(r"\'ll", " will ", text)
    text = re.sub('\W', ' ', text)
    text = re.sub('\s+', ' ', text)

    text = text.split()
    text = " ".join(text)

    return text
data['odds_free'] = data['text'].apply(lambda x: remove_odd(x.lower()))
display(data.head())

,unnamed: 0,text,toxic,odds_free
0,0,Explanation\nWhy the edits made under my usern...,0,explanation why the edits made under my userna...
1,1,D'aww! He matches this background colour I'm s...,0,d aww he matches this background colour i m se...
2,2,"Hey man, I'm really not trying to edit war. It...",0,hey man i m really not trying to edit war it s...
3,3,"""\nMore\nI can't make any real suggestions on ...",0,more i can t make any real suggestions on impr...
4,4,"You, sir, are my hero. Any chance you remember...",0,you sir are my hero any chance you remember wh...


<font color='blue'><b>Комментарий ревьюера: </b></font> ✔️\
<font color='green'>Очистка сделана верно! Мы оставили только символы Латинского алфавита и привели к нижнему регистру!\
Убрали частые сокращения.</font>

In [32]:
# удалим стоп слова
stop_words = set(nltk_stopwords.words('english'))

def remove_stopwords(text):
    text_ready = [i for i in text.split() if not i in stop_words]
    return text_ready

data['stopwords_free'] = data['odds_free'].apply(lambda x: remove_stopwords(x))
display(data.head())

,unnamed: 0,text,toxic,odds_free,stopwords_free
0,0,Explanation\nWhy the edits made under my usern...,0,explanation why the edits made under my userna...,"[explanation, edits, made, username, hardcore,..."
1,1,D'aww! He matches this background colour I'm s...,0,d aww he matches this background colour i m se...,"[aww, matches, background, colour, seemingly, ..."
2,2,"Hey man, I'm really not trying to edit war. It...",0,hey man i m really not trying to edit war it s...,"[hey, man, really, trying, edit, war, guy, con..."
3,3,"""\nMore\nI can't make any real suggestions on ...",0,more i can t make any real suggestions on impr...,"[make, real, suggestions, improvement, wondere..."
4,4,"You, sir, are my hero. Any chance you remember...",0,you sir are my hero any chance you remember wh...,"[sir, hero, chance, remember, page]"


<font color='blue'><b>Комментарий ревьюера: </b></font> ✔️\
<font color='green'> Убрали частые неинформативные слова!</font>

In [33]:
# определение тега POS
def get_wordnet_pos(word):
    """Map POS tag to first character lemmatize() accepts"""
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

In [34]:
%%time
# выполним лемматизацию
lemmatizer = nltk.WordNetLemmatizer()

def lemmatize_text(text):
    lemmatized_text = ' '.join([lemmatizer.lemmatize(x, get_wordnet_pos(x)) for x in nltk.word_tokenize(text)])
    return lemmatized_text

data['lemmatized'] = data['odds_free'].apply(lambda x: lemmatize_text(x))
display(data.head())

,unnamed: 0,text,toxic,odds_free,stopwords_free,lemmatized
0,0,Explanation\nWhy the edits made under my usern...,0,explanation why the edits made under my userna...,"[explanation, edits, made, username, hardcore,...",explanation why the edits make under my userna...
1,1,D'aww! He matches this background colour I'm s...,0,d aww he matches this background colour i m se...,"[aww, matches, background, colour, seemingly, ...",d aww he match this background colour i m seem...
2,2,"Hey man, I'm really not trying to edit war. It...",0,hey man i m really not trying to edit war it s...,"[hey, man, really, trying, edit, war, guy, con...",hey man i m really not try to edit war it s ju...
3,3,"""\nMore\nI can't make any real suggestions on ...",0,more i can t make any real suggestions on impr...,"[make, real, suggestions, improvement, wondere...",more i can t make any real suggestion on impro...
4,4,"You, sir, are my hero. Any chance you remember...",0,you sir are my hero any chance you remember wh...,"[sir, hero, chance, remember, page]",you sir be my hero any chance you remember wha...


CPU times: user 5min 4s, sys: 31.2 s, total: 5min 35s
Wall time: 5min 36s


<font color='blue'><b>Комментарий ревьюера: </b></font> ✔️\
<font color='green'>Здорово что выводишь данные, так удобно отлаживать код, сразу видно, как работает функция.</font>

<font color='blue'><b>Комментарий ревьюера: </b></font> ❌\
<font color='red'>Обрати внимание, не все слова приведены к начальным формам. Чтобы корректно обработались все части речи, для WordNetLemmatizer() нужно использовать POS-теги (Part of Speech, части речи). Примеры работы с WordNetLemmatizer(), а также с другими инструментами для лемматизации, можно найти в [этой статье](https://webdevblog.ru/podhody-lemmatizacii-s-primerami-v-python/)</font>

<div class="alert alert-block alert-warning">
<b>Комментарий студента:</b> <br>
<b>Изменения:</b> Были внесены следующие изменения: использованы POS-теги
</div>


In [35]:
df = data[['toxic', 'lemmatized']]
print('Количество дубликатов -', df.duplicated().sum())
df = df.drop_duplicates()
print('Количество дубликатов -', df.duplicated().sum())

Количество дубликатов - 1312
Количество дубликатов - 0


<font color='blue'><b>Комментарий ревьюера: </b></font> ⚠️\
<font color='darkorange'> Это можно добавить в функцию Лемматизации.</font>

<div class="alert alert-block alert-warning">
<b>Комментарий студента:</b> <br>
<b>Изменения:</b> Были внесены следующие изменения: перенесено в функцию Лемматизации
</div>

In [36]:
# разобьем выборку
X = df['lemmatized']
y = df['toxic']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state=42)

display(X_train.shape)
display(X_test.shape)

(110586,)

(47394,)

<font color='blue'><b>Комментарий ревьюера: </b></font> ✔️\
<font color='green'> Здорово , что у нас есть выборка для тестов!</font>

Создадим векторы, с параметрами: <br>
количество признаков ограничим 15000; <br>
минимальное количество встречаемости слов - 5,  <br>
ограничим встречаемость слов выше 70 % <br>
удалим диакритические знаки <br>

In [37]:
# без n-grams
vec = TfidfVectorizer(stop_words='english', strip_accents='unicode',
                      max_features=15000,
                      min_df=5,
                      max_df=.7
                      )
# n-grams
vec_ng = TfidfVectorizer(stop_words='english', strip_accents='unicode',
                         max_features=15000,
                         min_df=5,
                         max_df=.7,
                         ngram_range=(1, 2),
                         binary=True
                         )

 <span style="color:#8b51f6">**Вывод**</span> &#x2705; <br>
 
&#x1F31F; данные очищены от пунктуации, текст приведен в нижний регистр <br>
&#x1F31F; удалены стоп слова<br> 
&#x1F31F; выполнена лемматизация <br>
&#x1F31F; данные разбиты на выборки  <br>
&#x1F31F; созданы предварительные векторы для обучения: с n-gramами и без

<font color='blue'><b>Комментарий ревьюера : </b></font> ✔️\
<font color='green'> 👍</font>

## Обучение

In [38]:
best_estimator, score, score_test = [], [], []

In [39]:
# общая функция для обучения моделей
def models_fit(model, params,
                        tfidf=vec,
                        features=X_train, target=y_train,
                        feature_test=X_test, target_test=y_test):

    steps = [('tfidf', tfidf), ('model', model)]
    pipe = Pipeline(steps)
    search = RandomizedSearchCV(estimator=pipe, param_distributions=params,
                              cv=10, n_jobs=-1, scoring='f1', random_state=5)
    search.fit(features, target)
    pred_test = search.best_estimator_.predict(feature_test)
    test_result = f1_score(y_test, pred_test, zero_division=0)

    print('Модель с лучшими гиперпараметрами:', search.best_estimator_)
    print(f'F1 на тренировочной выборке {search.best_score_:.3f}'.format())
    print(f'F1 на тестовой выборке {test_result:.3f}'.format())

    best_estimator.append(search.best_estimator_)
    score.append(search.best_score_)
    score_test.append(test_result)

    return search.best_estimator_, search.best_score_

<font color='blue'><b>Комментарий ревьюера: </b></font> ✔️\
<font color='green'>Классно, что используешь pipeline. Так можно избежать утечек даже при кроссвалидации моделей. \
С Pipeline можно подбирать гиперпараметры не только к классификатору, но и к предобработчикам.</font>

<font color='blue'><b>Комментарий ревьюера : </b></font> ✔️\
<font color='green'>Подбор гиперпараметров проведён верно.</font>

<font color='blue'><b>Комментарий ревьюера: </b></font> ✔️\
<font color='green'>Здорово, что есть оценка кроссвалидацией.</font>

<span style="color:#fc4f80">**LogisticRegression**</span> &#x2705; <br>

In [40]:
lg_params = {
    'model__C': [.1, 1, 2, 3, 5, 10],
    'model__class_weight': ['balanced', None],
    'model__max_iter': [1000, 2000, 3000]
}

In [41]:
%%time
lg, lg_f1 = models_fit(model=LogisticRegression(random_state=5, n_jobs=-1),
                                  params=lg_params,
                                  tfidf=vec)

Модель с лучшими гиперпараметрами: Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.7, max_features=15000, min_df=5,
                                 stop_words='english',
                                 strip_accents='unicode')),
                ('model',
                 LogisticRegression(C=3, max_iter=2000, n_jobs=-1,
                                    random_state=5))])
F1 на тренировочной выборке 0.759
F1 на тестовой выборке 0.759
CPU times: user 5.82 s, sys: 2.2 s, total: 8.01 s
Wall time: 45.1 s


In [42]:
%%time
lg_ngram, lg_ngram_f1 = models_fit(model=LogisticRegression(random_state=5, n_jobs=-1),
                                  params=lg_params,
                                  tfidf=vec_ng)

Модель с лучшими гиперпараметрами: Pipeline(steps=[('tfidf',
                 TfidfVectorizer(binary=True, max_df=0.7, max_features=15000,
                                 min_df=5, ngram_range=(1, 2),
                                 stop_words='english',
                                 strip_accents='unicode')),
                ('model',
                 LogisticRegression(C=3, max_iter=2000, n_jobs=-1,
                                    random_state=5))])
F1 на тренировочной выборке 0.756
F1 на тестовой выборке 0.751
CPU times: user 10.7 s, sys: 2.66 s, total: 13.3 s
Wall time: 2min 1s


<font color='blue'><b>Комментарий ревьюера : </b></font> ✔️\
<font color='green'> 👍</font>

<span style="color:#fc4f80">**LGBMClassifier**</span> &#x2705; <br>

In [43]:
params_lgbm = {'model__boosting_type': ['gbdt', 'dart'],
               'model__num_leaves': [30, 35, 40],
               'model__n_estimators': [100, 150],
               'model__class_weight': ['balanced', None]}

In [44]:
%%time
lbgm, f1_lbgm = models_fit(model=lgb.LGBMClassifier(n_jobs=-1, random_state=42),
                                  params=params_lgbm,
                                  tfidf=vec)

[LightGBM] [Info] Number of positive: 10064, number of negative: 89463
[LightGBM] [Info] Number of positive: 10064, number of negative: 89463
[LightGBM] [Info] Number of positive: 10064, number of negative: 89463
[LightGBM] [Info] Number of positive: 10064, number of negative: 89463
[LightGBM] [Info] Number of positive: 10063, number of negative: 89464
[LightGBM] [Info] Number of positive: 10063, number of negative: 89464
[LightGBM] [Info] Number of positive: 10064, number of negative: 89464
[LightGBM] [Info] Number of positive: 10064, number of negative: 89464
[LightGBM] [Info] Number of positive: 10064, number of negative: 89464
[LightGBM] [Info] Number of positive: 10064, number of negative: 89464
[LightGBM] [Info] Number of positive: 10064, number of negative: 89463
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 5.909038 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 432749
[LightGBM] [Info] Num

In [45]:
%%time
lbgm_ng, f1_lbgm_ng = models_fit(model=lgb.LGBMClassifier(n_jobs=-1, random_state=42),
                                  params=params_lgbm,
                                  tfidf=vec_ng)

[LightGBM] [Info] Number of positive: 10064, number of negative: 89463
[LightGBM] [Info] Number of positive: 10063, number of negative: 89464
[LightGBM] [Info] Number of positive: 10064, number of negative: 89463
[LightGBM] [Info] Number of positive: 10064, number of negative: 89463
[LightGBM] [Info] Number of positive: 10064, number of negative: 89463
[LightGBM] [Info] Number of positive: 10064, number of negative: 89464
[LightGBM] [Info] Number of positive: 10063, number of negative: 89464
[LightGBM] [Info] Number of positive: 10064, number of negative: 89464
[LightGBM] [Info] Number of positive: 10064, number of negative: 89463
[LightGBM] [Info] Number of positive: 10064, number of negative: 89464
[LightGBM] [Info] Number of positive: 10064, number of negative: 89464
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 12.848428 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise

<font color='blue'><b>Комментарий ревьюера : </b></font> ✔️\
<font color='green'> 👍</font>

## Выводы

In [46]:
models = ['LogisticRegression', 'LogisticRegression_ngramm',
               'LGBMClassifier', 'LGBMClassifier_ngramm']
df_new = pd.DataFrame({'best_score_':score, 'f1_test': score_test}, index=models).sort_values(by='f1_test',  ascending=False)
display(df_new)

,best_score_,f1_test
LogisticRegression,0.759410,0.758702
LGBMClassifier,0.757222,0.758252
LGBMClassifier_ngramm,0.754195,0.754044
LogisticRegression_ngramm,0.756339,0.750771


<font color='blue'><b>Комментарий ревьюера: </b></font> ✔️\
<font color='green'> Отлично, что есть табличка результатов для сравнения.</font>

В ходе раобты были обучены и протестированы 4 модели: LogisticRegression', 'LogisticRegression_ngramm', 'LGBMClassifier', 'LGBMClassifier_ngramm' с n-grammами и без. <br>
Были получены следующие результаты: 


| **model**| **best_score_**|**f1_test**|
|:-----|:-----|:-----|
|LogisticRegression|0.759410|0.758702|
|LogisticRegression_ngramm|0.757222	|0.758252|
|LGBMClassifier_ngramm|	0.754195|0.754044|
|LGBMClassifier|0.756339|0.750771|



<style>
mark{
    color:red;
}
</style>
    
 &#x1F31F; Лучшие показатели у модели  <mark>**LogisticRegression без n-gramm**</mark> с гиперпараметрами (C=3, max_iter=2000, n_jobs=-1, random_state=5))]) - <mark>**F1 0.759**</mark>, что соотвествует запросу заказчика.
Худшая точность предсказаний у модели LGBMClassifier F1 0.756 на кросс-валидации.

<font color='blue'><b>Комментарий ревьюера: </b></font> ❌\
<font color='red'> Модель нужно выбрать по результатам кросс-валидации. Тетсовая выборка - для подтверждения результатов и контроля переобучения моделей.</font>

<div class="alert alert-block alert-warning">
<b>Комментарий студента:</b> <br>
<b>Изменения:</b> Были внесены следующие изменения: модель выбрана по результатам кросс-валидации
</div>

<font color='blue'><b>Итоговый комментарий ревьюера</b></font>\
<font color='green'>Оксана, хороший проект получился!
Большое спасибо за проделанную работу. Видно, что приложено много усилий.
</font>

<font color='blue'>Что нужно исправить:</font>
<ul><font color='red'>Поправь Лемматизацию.</font></ul>
<ul><font color='red'>Выбери Лучшую модель по кроссвалидации и проведи её тесты на тестовых данных.</font></ul>

<font color='blue'>Что можно сделать лучше:</font>
<font color='orange'>В работе я оставил несколько советов. Буду рад, если ты учтешь их.</font></ul>

<font color='blue'><b>Жду новую версию проекта :)</b></font>

## Чек-лист проверки

- [x]  Jupyter Notebook открыт
- [x]  Весь код выполняется без ошибок
- [x]  Ячейки с кодом расположены в порядке исполнения
- [x]  Данные загружены и подготовлены
- [x]  Модели обучены
- [x]  Значение метрики *F1* не меньше 0.75
- [x]  Выводы написаны